<a href="https://colab.research.google.com/github/vinay-852/GitHubIssueTriage/blob/docker/Sample_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://huggingface.co/spaces/vinay-pepakayala/GitHubIssueTriageManager

  Cloning https://huggingface.co/spaces/vinay-pepakayala/GitHubIssueTriageManager to /tmp/pip-req-build-a0dj2976
  Running command git clone --filter=blob:none --quiet https://huggingface.co/spaces/vinay-pepakayala/GitHubIssueTriageManager /tmp/pip-req-build-a0dj2976
  fatal: error reading section header 'shallow-info'
  Resolved https://huggingface.co/spaces/vinay-pepakayala/GitHubIssueTriageManager to commit d2cd79194c3ed5957df2ecc5d524de0dceffe596
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.5/705.5 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━

In [ ]:
from __future__ import annotations

import json
import os
from typing import Any, Dict, List

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import TypeAdapter, ValidationError

from GitHubIssueTriage import Action, ActionType

load_dotenv()

API_KEY = ""
BASE_URL = ""

ACTION_ADAPTER = TypeAdapter(Action)

SYSTEM_PROMPT = """
You are a GitHub Issue Triage Manager agent operating inside a strict Pydantic-validated action environment.

You MUST output exactly one JSON object and nothing else.

Hard requirements:
- The JSON object must conform to exactly one of the Action models in the discriminated union.
- The action discriminator key is "type", not "action".
- Never output plain strings, nested wrappers, explanations, markdown, or extra keys.
- Every action must include all required fields for that action.
- Use only these exact action type values:
  read_issue
  read_repo_rules
  read_label_definitions
  read_team_routing
  read_assignee_pool
  read_milestones
  search_similar_issues
  add_label
  remove_label
  assign_user
  set_priority
  set_milestone
  comment
  request_info
  provide_info
  mark_duplicate
  close_issue
  reopen_issue
  noop

Field requirements:
- read_issue MUST include "issue_id"
- request_info MUST include "fields"
- provide_info MUST include "fields"
- add_label MUST include "label"
- remove_label MUST include "label"
- assign_user MUST include "username"
- set_priority MUST include "priority"
- set_milestone MUST include "milestone"
- comment MUST include "text"
- mark_duplicate MUST include "issue_id"
- close_issue MUST include "reason"
- reopen_issue MUST include "reason"

Context rules:
- Read the issue and repo rules early if you need context before taking a triage action.
- If required information is missing, choose REQUEST_INFO and list only the missing fields.
- If a strong duplicate is present, choose MARK_DUPLICATE with the correct issue_id.
- Follow repo rules for labels, priority, milestone, severity, routing, and closure.
- Take only one action per step.
- Keep COMMENT short and focused.
""".strip()


class IssueTriageAgent:
    def __init__(self) -> None:
        self.api_base_url = BASE_URL
        self.api_key = API_KEY
        self.model_name = os.getenv("MODEL_NAME", "oca/gpt-5")
        self.temperature = float(os.getenv("TEMPERATURE", "0.2"))
        self.max_tokens = int(os.getenv("MAX_OUTPUT_TOKENS", "200"))

        print(f"Using API base URL: {self.api_base_url}")
        print(f"Using model: {self.model_name}")
        print(f"Using API key: {'Yes' if self.api_key else 'No'}")

        if not self.api_key:
            raise RuntimeError("Missing API_KEY / HF_TOKEN / OPENAI_API_KEY.")

        self.client = OpenAI(api_key=self.api_key, base_url=self.api_base_url)

    def _build_messages(self, observation: Dict[str, Any]) -> List[Dict[str, str]]:
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "instruction": "Choose the next best triage action.",
                        "observation": observation,
                    },
                    indent=2,
                ),
            },
        ]

    def _strip_code_fences(self, text: str) -> str:
        text = text.strip()
        if text.startswith("```"):
            lines = text.splitlines()
            if len(lines) >= 2 and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            text = "\n".join(lines).strip()
        return text

    def _extract_json_object(self, text: str) -> str:
        text = self._strip_code_fences(text)

        if text.startswith("{") and text.endswith("}"):
            return text

        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return text[start : end + 1]

        raise ValueError("No JSON object found in model output.")

    def _sanitize_action_dict(self, data: Dict[str, Any]) -> Dict[str, Any]:
        data = dict(data)

        if "type" not in data:
            if "action" in data:
                data["type"] = data.pop("action")
            elif "action_type" in data:
                data["type"] = data.pop("action_type")

        payload = data.pop("action_payload", None)
        if isinstance(payload, dict):
            for key, value in payload.items():
                data.setdefault(key, value)

        for key in [
            "outcome",
            "success",
            "timestamp",
            "step_index",
            "message",
            "analysis",
            "thought",
            "reasoning",
        ]:
            data.pop(key, None)

        return data

    def _parse_action_json(self, raw_text: str) -> Dict[str, Any]:
        json_text = self._extract_json_object(raw_text)
        data = json.loads(json_text)
        if not isinstance(data, dict):
            raise ValueError("Model output was not a JSON object.")

        sanitized = self._sanitize_action_dict(data)
        action = ACTION_ADAPTER.validate_python(sanitized)
        return action.model_dump(mode="json", exclude_none=True)

    def next_action(self, observation: Dict[str, Any]) -> Dict[str, Any]:
        try:
            stream = self.client.chat.completions.create(
                model=self.model_name,
                messages=self._build_messages(observation),
                stream=True,
                temperature=self.temperature,
                max_tokens=self.max_tokens,
            )

            parts: List[str] = []
            for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    parts.append(delta)

            raw_text = "".join(parts).strip()
            print(f"[Debug] LLM Raw Stream Output: {raw_text}")

            return self._parse_action_json(raw_text)

        except Exception as e:
            print(f"Error occurred in next_action_streaming: {e}")
            return {"type": ActionType.NOOP.value}

In [ ]:
from __future__ import annotations

import argparse
import json
from typing import Any, Dict, List, Optional, Protocol
from types import SimpleNamespace
from urllib.parse import urlparse

try:
    from GitHubIssueTriage import (
        ActionType,
        GithubissuetriageEnv,
        grade_episode,
        load_episode_from_source,
        GitHubIssueTriageEnvironment,
    )
except ImportError:  # pragma: no cover
    from models import ActionType
    from client import GithubissuetriageEnv
    from server.grader import grade_episode
    from server.loader import load_episode_from_source


def _structured_print(label: str, payload: Dict) -> None:
    print(f"[{label}] {json.dumps(payload, sort_keys=True)}", flush=True)


def _observation_snapshot(observation) -> Dict:
    return {
        "step_count": observation.step_count,
        "remaining_steps": observation.remaining_steps,
        "done": observation.done,
        "last_action_message": observation.last_action_message,
        "objective_summary": observation.objective_summary,
        "pending_missing_fields": getattr(observation, "pending_missing_fields", []),
        "provided_fields": getattr(observation, "provided_fields", {}),
    }


def _parse_github_issue_url(issue_url: str) -> tuple[str, str]:
    parsed = urlparse(issue_url)
    parts = [p for p in parsed.path.split("/") if p]

    if len(parts) < 4 or parts[2] != "issues":
        raise ValueError(
            f"Unsupported GitHub issue URL: {issue_url}. Expected https://github.com/OWNER/REPO/issues/123"
        )

    owner = parts[0]
    repo = parts[1]
    issue_id = parts[3]
    return f"{owner}/{repo}", issue_id


def _collect_provided_info(fields: List[str]) -> Dict[str, str]:
    print("\nThe environment requested more information.")
    print("Enter values for the fields below. Leave blank to skip any field.\n")

    values: Dict[str, str] = {}
    for field in fields:
        value = input(f"{field}: ").strip()
        if value:
            values[field] = value
    return values


def _build_provide_info_action(fields: List[str]) -> Dict[str, object]:
    return {
        "type": ActionType.PROVIDE_INFO.value,
        "fields": _collect_provided_info(fields),
    }


class EpisodeEnv:
    """Adapter that returns an observation from reset(), which run_episode expects."""

    def __init__(self, sync_client):
        self._client = sync_client
        self._last_observation = None

    def __enter__(self):
        self._client.__enter__()
        return self

    def __exit__(self, exc_type, exc, tb):
        return self._client.__exit__(exc_type, exc, tb)

    def reset(self, **kwargs):
        result = self._client.reset(**kwargs)
        self._last_observation = result.observation
        return self._last_observation

    def step(self, action, **kwargs):
        result = self._client.step(action, **kwargs)
        reward_value = float(result.reward or 0.0)
        self._last_observation = result.observation

        reward_proxy = SimpleNamespace(
            total=reward_value,
            model_dump=lambda: {"total": reward_value},
        )
        info_proxy = SimpleNamespace(
            action_valid=getattr(result.observation, "last_action_valid", True),
            action_effect=getattr(result.observation, "last_action_message", ""),
            grader_notes=[],
            changed_fields=[],
            reward_breakdown={"total": reward_value},
            reward_components={},
        )

        return SimpleNamespace(
            observation=result.observation,
            reward=reward_proxy,
            done=result.done,
            info=info_proxy,
        )

    @property
    def state(self):
      if self._last_observation is None:
        return SimpleNamespace()
      return self._last_observation


def run_episode(
    env,
    agent: IssueTriageAgent,
) -> Dict[str, float]:
    observation = env.reset()

    _structured_print(
        "START",
        {
            "task_id": observation.task.task_id,
            "episode_id": observation.episode_id,
            "difficulty": observation.task.difficulty.value,
            "max_steps": observation.task.max_steps,
            "objective_summary": observation.objective_summary,
        },
    )

    step_index = 0

    while True:
        action = agent.next_action(observation.model_dump())

        # Debug logging for the action being sent
        _structured_print("DEBUG_ACTION", action)

        try:
            step_result = env.step(action)
        except RuntimeError as e:
            # Catch OpenEnv server errors and provide context
            error_msg = str(e)
            print(f"[ERROR] Environment step failed: {error_msg}", flush=True)
            print(f"[ERROR] Action was: {json.dumps(action, sort_keys=True)}", flush=True)
            print(f"[ERROR] Observation state: episode_id={observation.episode_id}, step={step_index}", flush=True)
            raise

        _structured_print(
            "STEP",
            {
                "task_id": observation.task.task_id,
                "step": step_index,
                "action": action,
                "reward": step_result.reward.total,
                "reward_breakdown": step_result.reward.model_dump(),
                "action_valid": step_result.info.action_valid,
                "action_effect": step_result.info.action_effect,
                "grader_notes": step_result.info.grader_notes,
                "observation": _observation_snapshot(step_result.observation),
            },
        )

        observation = step_result.observation
        step_index += 1

        if step_result.done:
            break

        if action.get("type") == ActionType.REQUEST_INFO.value:
            requested_fields = action.get("fields") or list(
                getattr(observation, "pending_missing_fields", [])
            )
            if requested_fields:
                provide_action = _build_provide_info_action(list(requested_fields))
                try:
                    provide_result = env.step(provide_action)
                except RuntimeError as e:
                    error_msg = str(e)
                    print(f"[ERROR] Environment step failed on PROVIDE_INFO: {error_msg}", flush=True)
                    print(f"[ERROR] Action was: {json.dumps(provide_action, sort_keys=True)}", flush=True)
                    raise

                _structured_print(
                    "STEP",
                    {
                        "task_id": observation.task.task_id,
                        "step": step_index,
                        "action": provide_action,
                        "reward": provide_result.reward.total,
                        "reward_breakdown": provide_result.reward.model_dump(),
                        "action_valid": provide_result.info.action_valid,
                        "action_effect": provide_result.info.action_effect,
                        "grader_notes": provide_result.info.grader_notes,
                        "observation": _observation_snapshot(provide_result.observation),
                    },
                )

                observation = provide_result.observation
                step_index += 1

                if provide_result.done:
                    break

    final_state = env.state
    grade = grade_episode(final_state)

    result_payload = {
        "task_id": observation.task.task_id,
        "steps_taken": step_index,
        "score": grade.score,
        "matched_labels": grade.matched_labels,
        "matched_assignee": grade.matched_assignee,
        "matched_priority": grade.matched_priority,
        "matched_milestone": grade.matched_milestone,
        "duplicate_matched": grade.duplicate_matched,
        "missing_fields_requested": grade.missing_fields_requested,
        "closed_correctly": grade.closed_correctly,
        "comment_accepted": grade.comment_accepted,
        "notes": grade.notes,
    }
    _structured_print("END", result_payload)

    return {"score": grade.score, "steps": float(step_index)}





def main() -> None:
    # 🔧 Hardcoded config (edit here if needed)
    repo_rules = "https://github.com/vinay-852/GitHubIssueTriage/blob/main/data/url_repo_rules.json"
    issue_url = "https://github.com/ai-joe-git/pocket-tts-server/issues/3"
    live_github = True
    task_id = None
    max_steps = 10
    base_url = "https://vinay-pepakayala-githubissuetriagemanager.hf.space"
    transport = "remote"
    episode = load_episode_from_source(
        repo_rules_path=repo_rules,
        issue_source=issue_url,
        live_github=live_github,
        task_id=task_id,
        max_steps=max_steps,
    )

    agent = IssueTriageAgent()

    if transport == "local":

        env = GitHubIssueTriageEnvironment(
            episodes=[episode],
            strict_mode=True,
            live_github=True,
        )
        run_episode(env, agent)
    else:
        with GithubissuetriageEnv(
            base_url=base_url,
            episodes=[episode],
            strict_mode=True,
            live_github=True,
        ).sync() as client:
            env = EpisodeEnv(client)
            print(run_episode(env, agent))

if __name__ == "__main__":
    main()

Using API base URL: https://code-internal.aiservice.us-chicago-1.oci.oraclecloud.com/20250206/app/litellm
Using model: oca/gpt-5
Using API key: Yes
[CLIENT_PARSE_RESULT] Received payload keys: ['observation', 'reward', 'done']
[CLIENT_PARSE_RESULT] Observation keys: ['episode_id', 'task', 'issue', 'repo_rules', 'available_labels', 'available_assignees', 'available_milestones', 'candidate_duplicates', 'action_history', 'pending_missing_fields', 'objective_summary', 'progress_metrics', 'provided_fields', 'remaining_steps', 'step_count', 'last_action_valid', 'last_action_message']
[START] {"difficulty": "easy", "episode_id": "ep_easy_routing", "max_steps": 8, "objective_summary": ["Labels needed: component:api, severity:critical, status:triaged"], "task_id": "triage_easy_api_p1"}
[Debug] LLM Raw Stream Output: {"type":"add_label","label":"component:api"}
[DEBUG_ACTION] {"label": "component:api", "type": "add_label"}
[CLIENT_STEP_PAYLOAD] {"type": "add_label", "label": "component:api"}
[CL